In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Felixdiamond/videoTranslator.git"
PROJECT_DIR = "/kaggle/working/videoTranslator"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import subprocess
import sys


def patch_file(path, replacements):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    for old, new in replacements:
        content = content.replace(old, new)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


subprocess.run(["apt-get", "update"], check=True)
subprocess.run([
    "apt-get", "install", "-y", "-q",
    "ffmpeg", "rubberband-cli", "sox",
    "mecab", "libmecab-dev", "mecab-ipadic-utf8",
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.47.1", "accelerate==1.13.0",
    "faster-whisper>=1.2.1", "demucs>=4.0.1",
    "pyrubberband>=0.3.0", "librosa", "soundfile",
    "pydub", "noisereduce", "moviepy", "gtts",
    "sentencepiece", "sacremoses",
    "fastapi", "uvicorn[standard]", "psutil", "tqdm",
    "numpy>=2.2.2", "numba>=0.64.0",
    "einops", "onnxruntime",
], check=True)

subprocess.run(["git", "clone", "https://github.com/m-bain/whisperX.git", "/tmp/whisperx"], check=True)
patch_file("/tmp/whisperx/pyproject.toml", [
    ("torch~=2.8.0", "torch>=2.8.0"),
    ("torchaudio~=2.8.0", "torchaudio>=2.8.0"),
])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/tmp/whisperx"], check=True)

subprocess.run(["git", "clone", "https://github.com/Felixdiamond/MeloTTS.git", "/tmp/MeloTTS"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "/tmp/MeloTTS"], check=True)
subprocess.run([sys.executable, "-m", "unidic", "download"], check=True)

# --- Qwen3-TTS (voice cloning) --- uncomment below if you want Qwen3-TTS instead of / in addition to MeloTTS
# subprocess.run(["git", "clone", "https://github.com/QwenLM/Qwen3-TTS.git", "/tmp/qwen3tts"], check=True)
# patch_file("/tmp/qwen3tts/pyproject.toml", [
#     ("transformers==4.57.3", "transformers>=4.47.1"),
# ])
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/tmp/qwen3tts"], check=True)

import torch
print("torch:", torch.__version__)
print("torchaudio:", __import__("torchaudio").__version__)


In [ ]:
import importlib

for name in ["numpy", "numba"]:
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    print(f"{name}: {mod.__version__}")

from melo.api import TTS
print("MeloTTS: OK")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("Torch:", torch.__version__)

In [ ]:
import os
import re
import subprocess

os.chdir("/kaggle/working/videoTranslator")

with open("config.yaml", "r", encoding="utf-8") as f:
    config = f.read()

config = re.sub(r"^hardware_tier:\s*.*$", "hardware_tier: gpu_high", config, flags=re.MULTILINE)

# Default: MeloTTS, no voice cloning
config = re.sub(r"^tts_mode:\s*.*$", "tts_mode: melo", config, flags=re.MULTILINE)
config = re.sub(r"^enable_voice_cloning:\s*.*$", "enable_voice_cloning: false", config, flags=re.MULTILINE)

# --- If you installed Qwen3-TTS and want voice cloning, comment the two lines above and uncomment these:
# config = re.sub(r"^tts_mode:\s*.*$", "tts_mode: qwen3", config, flags=re.MULTILINE)
# config = re.sub(r"^enable_voice_cloning:\s*.*$", "enable_voice_cloning: true", config, flags=re.MULTILINE)

with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(config)

result = subprocess.run(
    ["grep", "-E", "hardware_tier|tts_mode|enable_voice_cloning", "config.yaml"],
    capture_output=True, text=True,
)
print("Config updated:\n" + result.stdout)


In [ ]:
import subprocess
import sys

video_path = "/kaggle/input/datasets/felixdiamond/source-vids/fern_eng_short.mp4"
target_lang = "fr"

subprocess.run([sys.executable, "translator.py", video_path, target_lang], check=True)